In [1]:
import numpy as np
import pandas as pd
import mlflow
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Modeling")

c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1785101030275, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785101030275, lifecycle_stage='active', name='Modeling', tags={}, trace_location=None, workspace='default'>

In [4]:
mlflow.sklearn.autolog()

path = "../data/processed/Bank_Churn_clean.csv"
df = pd.read_csv(path)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
TARGET = "Exited"
X = df.drop(columns=[TARGET])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = 0.2, stratify= y, random_state=42)

In [6]:

ohe = OneHotEncoder( handle_unknown='ignore')
cat_cols = ['Gender', 'Geography']

ohe.fit(X_train[cat_cols])
X_train_ohe = ohe.transform(X_train[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)
X_train[ohe_cols] = pd.DataFrame(X_train_ohe.toarray(), columns=ohe_cols, index=X_train.index)
X_test[ohe_cols] = pd.DataFrame(X_test_ohe.toarray(), columns=ohe_cols, index=X_test.index)

X_train = X_train.drop(columns=cat_cols)
X_test = X_test.drop(columns=cat_cols)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
with mlflow.start_run():
    dt = DecisionTreeClassifier(max_depth=10,class_weight="balanced")
    dt.fit(X_train_scaled, y_train)

    y_pred = dt.predict(X_test_scaled)
    y_proba = dt.predict_proba(X_test_scaled)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("F1-Score:", f1_score(y_test, y_pred))

2026/07/27 01:09:28 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/07/27 01:09:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Accuracy: 0.7475
Recall: 0.683046683046683
Precision: 0.42507645259938837
F1-Score: 0.5240339302544769
🏃 View run unruly-doe-375 at: http://localhost:5000/#/experiments/3/runs/f6d90003917a41dbae8d1bb3059f4900
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [7]:
with mlflow.start_run():
    rf = RandomForestClassifier(max_depth=10,class_weight="balanced")
    rf.fit(X_train_scaled, y_train)

    y_pred = rf.predict(X_test_scaled)
    y_proba = rf.predict_proba(X_test_scaled)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("F1-Score:", f1_score(y_test, y_pred))

2026/07/27 11:03:57 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/07/27 11:03:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Accuracy: 0.816
Recall: 0.7051597051597052
Precision: 0.5364485981308411
F1-Score: 0.6093418259023354
🏃 View run zealous-auk-581 at: http://localhost:5000/#/experiments/3/runs/8744297deace40d99d53a0c2cd627035
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [30]:
mlflow.xgboost.autolog()
xgb = XGBClassifier(max_depth= 5, n_estimators=20)
xgb.fit(X_train_scaled, y_train)

y_pred = xgb.predict(X_test_scaled)
y_proba = xgb.predict_proba(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))

2026/07/27 01:09:45 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '68310885d5a34d939314401971a6e286', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow
2026/07/27 01:09:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run capricious-seal-695 at: http://localhost:5000/#/experiments/3/runs/68310885d5a34d939314401971a6e286
🧪 View experiment at: http://localhost:5000/#/experiments/3
Accuracy: 0.865
Recall: 0.4668304668304668
Precision: 0.7818930041152263
F1-Score: 0.5846153846153846


In [34]:
joblib.dump({"model:rf"}, "../models/rf_best.pkl")

['../models/rf_best.pkl']

In [ ]:
scores = cross_val_score(rf, X_train_scaled, y_train, cv=5) 

2026/07/27 11:04:32 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd8c61d7c952241cfb282ce1079ca5fbc', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 11:04:32 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/07/27 11:04:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1

🏃 View run persistent-stork-133 at: http://localhost:5000/#/experiments/3/runs/d8c61d7c952241cfb282ce1079ca5fbc
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 11:04:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/27 11:04:47 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '73d62c39f23642db8b299ad11c8e6d7d', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 11:04:47 WARNING mlflow.sklearn: Failed to log training dataset information

🏃 View run luxuriant-mouse-887 at: http://localhost:5000/#/experiments/3/runs/b97c88162ca34dac895476446e1544f9
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 11:04:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/27 11:04:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4bd9eeb0f2ce4b8e9cdee25d83b6c6ed', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 11:04:55 WARNING mlflow.sklearn: Failed to log training dataset information

🏃 View run placid-koi-214 at: http://localhost:5000/#/experiments/3/runs/73d62c39f23642db8b299ad11c8e6d7d
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 11:04:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/27 11:05:05 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2d6ad0edda8448bf9e97d5bd03d90c61', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 11:05:05 WARNING mlflow.sklearn: Failed to log training dataset information

🏃 View run carefree-bee-639 at: http://localhost:5000/#/experiments/3/runs/4bd9eeb0f2ce4b8e9cdee25d83b6c6ed
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 11:05:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.datascience\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run magnificent-wolf-520 at: http://localhost:5000/#/experiments/3/runs/2d6ad0edda8448bf9e97d5bd03d90c61
🧪 View experiment at: http://localhost:5000/#/experiments/3
0.826875


In [10]:
scores.mean(), scores.std()

(np.float64(0.826875), np.float64(0.0030362394503727834))